# 1.Import library that needed for this cases 

In [1]:
import pandas as pd
import numpy as np
import csv
import warnings
import os
import glob
from datetime import datetime
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Tuple, List

from scipy import sparse
from scipy.sparse import csr_matrix, save_npz
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score, confusion_matrix

from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import TomekLinks  # (opsional; default OFF)
from xgboost import XGBClassifier

RDLogger.DisableLog('rdApp.*')
ignore_warnings = True
if ignore_warnings:
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


# 2.load the dataset from sider csv 


In [2]:
Data_path = '/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/SIDER/sider.csv'

df_imbalanced = pd.read_csv(
    Data_path,
    sep=',',
    engine='python',
)
print("Data loaded successfully.")

Data loaded successfully.


# 3.Convert the smile into morgan fingerprint , check The Ir for each column

In [3]:
assert 'smiles' in df_imbalanced.columns, "Column 'smiles' not found in the dataset." # Ensure 'smiles' column exists for fingerprint generation

#define morgan functions to convert smiles to morgan fingerprints
def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        arr = np.zeros((1,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr
    else:
        return np.zeros((n_bits,), dtype=np.int8)


#extract all smiles to CSR matrix uint8 
X_arr = np.array([smiles_to_morgan_fp(smiles) for smiles in df_imbalanced['smiles']])
X_int = sparse.csr_matrix(X_arr, dtype=np.uint8)

# --- Groups (anti-leak) ---
Groups = df_imbalanced['group_id'].values if 'group_id' in df_imbalanced.columns else np.arange(len(df_imbalanced))

# check ir lowest, median, and highest for each label 3 organs
label_cols = [col for col in df_imbalanced.columns if col not in ['smiles', 'group_id','char']]

#binary detection of imbalance ratio for each label

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.Series(s).dropna().astype(int).unique().tolist()
    return set(vals).issubset({0, 1}) and len(vals) >= 1

label_cols = [c for c in df_imbalanced.columns if c not in ['smiles', 'group_id','char'] and is_binary_series(df_imbalanced[c])]
assert len(label_cols) >= 1, "Tidak ada kolom label biner 0/1 yang terdeteksi."


# Calculate imbalance ratio for each label and sort labels by imbalance ratio
def imbalance_ratio(label):
    pos = df_imbalanced[label].sum()
    neg = len(df_imbalanced) - pos
    if neg == 0 or pos == 0:
        return float('inf')  # Handle case where all samples belong to one class
    return max(pos, neg) / min(pos, neg)

sorted_labels = sorted(label_cols, key=imbalance_ratio, reverse=True)
if len(sorted_labels) >= 9:
    largest_3_labels = sorted_labels[:3]
    mid_idx = len(sorted_labels) // 2
    median_3_labels = sorted_labels[mid_idx-1 : mid_idx+2]
    smallest_3_labels = sorted_labels[-3:]
    labels_selected = largest_3_labels + median_3_labels + smallest_3_labels
else:
    labels_selected = sorted_labels

def _ir_str(lbl):
    val = imbalance_ratio(lbl)
    return "inf" if np.isinf(val) else f"{val:.3f}"

print("Imbalance ratios calculated successfully.")

print("SELECTED_LABELS (3 largest, 3 median, 3 smallest):")
for label in labels_selected:
    print(f"  - {label} (IR={_ir_str(label)})")

Imbalance ratios calculated successfully.
SELECTED_LABELS (3 largest, 3 median, 3 smallest):
  - Product issues (IR=63.864)
  - Skin and subcutaneous tissue disorders (IR=12.092)
  - Nervous system disorders (IR=10.602)
  - Respiratory, thoracic and mediastinal disorders (IR=2.888)
  - Neoplasms benign, malignant and unspecified (incl cysts and polyps) (IR=2.795)
  - Immune system disorders (IR=2.541)
  - Ear and labyrinth disorders (IR=1.165)
  - Hepatobiliary disorders (IR=1.086)
  - Reproductive system and breast disorders (IR=1.039)


# 4. validate the dataset and check the datatype for each Column 

In [4]:
#showing the info of the dataset, shape, and first 30 columns
print("=== INFO DATA ===")
df_imbalanced.info()
print("\n=== SHAPE ===", df_imbalanced.shape)
print("\n=== KOLOM (awal) ===", df_imbalanced.columns.tolist()[:30])

# Drop baris tanpa SMILES valid
before = len(df_imbalanced)
df_imbalanced = df_imbalanced[df_imbalanced["smiles"].astype(str).str.len() > 0].copy()
print(f"[INFO] Hapus {before - len(df_imbalanced)} baris SMILES kosong/invalid.")

=== INFO DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1427 entries, 0 to 1426
Data columns (total 28 columns):
 #   Column                                                               Non-Null Count  Dtype 
---  ------                                                               --------------  ----- 
 0   smiles                                                               1427 non-null   object
 1   Hepatobiliary disorders                                              1427 non-null   int64 
 2   Metabolism and nutrition disorders                                   1427 non-null   int64 
 3   Product issues                                                       1427 non-null   int64 
 4   Eye disorders                                                        1427 non-null   int64 
 5   Investigations                                                       1427 non-null   int64 
 6   Musculoskeletal and connective tissue disorders                      1427 non-null   int64 
 7

# 5.Detect the binary label 0/1 in each column 

In [5]:
#checking for binary labels (0/1) and converting them to numeric if necessary
NON_LABELS = ["smiles", "char", "group_id"]

def detect_binary_labels(df, non_labels):
    cand = [c for c in df.columns if c not in non_labels]
    label_cols, skipped = [], []
    map_bool = {"true":1, "false":0, "y":1, "n":0, "yes":1, "no":0} # mapping for common boolean strings
    for c in cand:
        s = df[c].dropna().astype(str).str.lower()
        uniq = set(s.unique())
        if uniq <= {"0", "1"}:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
            label_cols.append(c)
        elif uniq <= set(list(map_bool.keys()) + ["0","1"]):
            df[c] = (
                df[c].astype(str).str.lower().map(map_bool)
                .fillna(pd.to_numeric(df[c], errors="coerce"))
                .fillna(0).astype(int)
            )
            label_cols.append(c)
        else:
            skipped.append(c)  # bukan biner → abaikan
    return df, label_cols, skipped

df_imbalanced, LABEL_COLS, SKIPPED_COLS = detect_binary_labels(df_imbalanced, NON_LABELS)
if not LABEL_COLS:
    raise ValueError("Tidak ditemukan kolom label biner 0/1.")
print(f"[OK] Jumlah label biner: {len(LABEL_COLS)}")
print("Contoh 10 label:", df_imbalanced[LABEL_COLS].columns.tolist()[:10])
if SKIPPED_COLS:
    print(f"[INFO] Kolom non-biner diabaikan (contoh): {df_imbalanced[SKIPPED_COLS].columns.tolist()[:10]}")

[OK] Jumlah label biner: 27
Contoh 10 label: ['Hepatobiliary disorders', 'Metabolism and nutrition disorders', 'Product issues', 'Eye disorders', 'Investigations', 'Musculoskeletal and connective tissue disorders', 'Gastrointestinal disorders', 'Social circumstances', 'Immune system disorders', 'Reproductive system and breast disorders']


# 6.from morgan 2048 bit to CSR uinnt 8 binary and save it into NPZ

In [6]:
#morgan fitures to CSR matrix uint8 (0/1), save to NPZ
def morgan_to_csr_uint8(smiles_list, n_bits=2048, radius=2,
                        use_chirality=True, use_bond_types=True, use_features=False):
    data, indices, indptr = [], [], [0]
    bad_idx = []
    for i, smi in enumerate(smiles_list):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            bad_idx.append(i); indptr.append(len(indices)); continue
        bv = AllChem.GetMorganFingerprintAsBitVect(
            m, radius=radius, nBits=n_bits,
            useChirality=use_chirality, useBondTypes=use_bond_types, useFeatures=use_features
        )
        onbits = bv.GetOnBits()
        indices.extend(onbits)
        data.extend([1]*len(onbits))  # integer 1
        indptr.append(len(indices))
    X_uint8 = csr_matrix((np.asarray(data, dtype=np.uint8),
                          np.asarray(indices, dtype=np.int32),
                          np.asarray(indptr, dtype=np.int32)),
                         shape=(len(smiles_list), n_bits), dtype=np.uint8)
    return X_uint8, bad_idx

MORGAN_BITS = 2048
MORGAN_RADIUS = 2
OUT_X_NPZ_INT = 'X_features_morgan_2048.npz'

smiles_list = df_imbalanced["smiles"].astype(str).tolist()
X_INT, bad_rows = morgan_to_csr_uint8(
    smiles_list, n_bits=MORGAN_BITS, radius=MORGAN_RADIUS,
    use_chirality=True, use_bond_types=True, use_features=False
)

assert X_INT.shape[1] == MORGAN_BITS, f"Panjang vektor = {X_INT.shape[1]}, harus {MORGAN_BITS}."
nnz = X_INT.nnz
tot = X_INT.shape[0] * X_INT.shape[1]
print(f"[OK] X_INT shape={X_INT.shape}, dtype={X_INT.dtype}, sparsity={1 - nnz/tot:.6f}")
if bad_rows:
    print(f"[WARN] {len(bad_rows)} SMILES invalid → baris nol. Contoh idx: {bad_rows[:10]}")

save_npz(OUT_X_NPZ_INT, X_INT)
print(f"[SAVE] NPZ fitur: {OUT_X_NPZ_INT}")

[OK] X_INT shape=(1427, 2048), dtype=uint8, sparsity=0.977223
[SAVE] NPZ fitur: X_features_morgan_2048.npz


# 7. Analyze the IR for each column dataset with 3 outcome label for each big/median/low 

In [9]:
# Analyze imbalance ratio (IR) for each label, save summary to CSV, and select 9 labels (3 terbesar, 3 median, 3 terkecil)
OUT_IMB_CSV = 'imbalance_summary.csv'

def ir_ratio(neg, pos):
    maj, mino = (neg, pos) if neg >= pos else (pos, neg)
    return float("inf") if mino == 0 else round(maj/mino, 6)

rows = []
n = len(df_imbalanced)
for c in LABEL_COLS:
    pos = int(df_imbalanced[c].sum()); neg = n - pos
    rows.append({"Label": c, "Negative":neg, "Positive":pos, "IR(maj/min)": ir_ratio(neg,pos), "Pos%": round(pos/n,4)})
imb = pd.DataFrame(rows).sort_values("IR(maj/min)", ascending=False).reset_index(drop=True)
imb.to_csv(OUT_IMB_CSV, index=False)
print(f"[SAVE] Ringkasan imbalance: {OUT_IMB_CSV}")

# To match the logic of choosing 3 largest, 3 median, 3 smallest labels from earlier
m = len(imb)
if m >= 9:
    largest = imb.iloc[:3]["Label"].tolist()
    mid_idx = m // 2
    median = imb.iloc[mid_idx-1 : mid_idx+2]["Label"].tolist()
    smallest = imb.iloc[-3:]["Label"].tolist()
    SELECTED_LABELS = list(dict.fromkeys(largest + median + smallest))  # unik, pertahankan urutan
else:
    SELECTED_LABELS = imb["Label"].tolist()

for lbl in SELECTED_LABELS:
    row = imb[imb["Label"] == lbl].iloc[0]
    print(f"  - {lbl}: Neg={row['Negative']}, Pos={row['Positive']}, IR={_ir_str(lbl)}")
print("[OK] 9 label terpilih (IR):", SELECTED_LABELS)

[SAVE] Ringkasan imbalance: imbalance_summary.csv
  - Product issues: Neg=1405, Pos=22, IR=63.864
  - Skin and subcutaneous tissue disorders: Neg=109, Pos=1318, IR=12.092
  - Nervous system disorders: Neg=123, Pos=1304, IR=10.602
  - Respiratory, thoracic and mediastinal disorders: Neg=367, Pos=1060, IR=2.888
  - Neoplasms benign, malignant and unspecified (incl cysts and polyps): Neg=1051, Pos=376, IR=2.795
  - Immune system disorders: Neg=403, Pos=1024, IR=2.541
  - Ear and labyrinth disorders: Neg=768, Pos=659, IR=1.165
  - Hepatobiliary disorders: Neg=684, Pos=743, IR=1.086
  - Reproductive system and breast disorders: Neg=700, Pos=727, IR=1.039
[OK] 9 label terpilih (IR): ['Product issues', 'Skin and subcutaneous tissue disorders', 'Nervous system disorders', 'Respiratory, thoracic and mediastinal disorders', 'Neoplasms benign, malignant and unspecified (incl cysts and polyps)', 'Immune system disorders', 'Ear and labyrinth disorders', 'Hepatobiliary disorders', 'Reproductive syst

# 8.Combined all the data to one csv 

In [10]:
OUT_COMBINED_CSV = 'combined_dataset.csv'

# Buat DataFrame sparse untuk fitur (0/1 integer)
bit_cols = [f"bit_{j}" for j in range(MORGAN_BITS)]
X_df = pd.DataFrame.sparse.from_spmatrix(X_INT, columns=bit_cols).astype(pd.SparseDtype("uint8"))

# Identitas (hanya 'smiles')
id_df = df_imbalanced[["smiles"]].reset_index(drop=True)

# Label terpilih
labels_df = df_imbalanced[SELECTED_LABELS].astype(int).reset_index(drop=True)

# Gabungkan
combined_df = pd.concat([id_df, X_df.reset_index(drop=True), labels_df], axis=1)

# Sanity: pastikan 2048 kolom bit
bit_cols_in_df = [c for c in combined_df.columns if c.startswith("bit_")]
assert len(bit_cols_in_df) == MORGAN_BITS, f"Kolom bit = {len(bit_cols_in_df)}, harus {MORGAN_BITS}."

combined_df.to_csv(OUT_COMBINED_CSV, index=False)
print(f"[SAVE] Dataset gabungan: {OUT_COMBINED_CSV}")

[SAVE] Dataset gabungan: combined_dataset.csv


# 8.compare the baseline withouth resampling 
- XGBOOST 
- RandomForest 


In [11]:
# ---------- Guards ----------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df_imbalanced" in globals() and "Groups" in globals(), "df_imbalanced/Groups belum ada (jalankan STEP 1)."
assert "SELECTED_LABELS" in globals(), "SELECTED_LABELS belum ada (jalankan STEP 5)."

RANDOM_STATE = 116
GROUPS = Groups

# ---------- Fitur untuk model (float32) ----------
X = X_INT.astype(np.float32)

# ---------- Metrik GM (Geometric Mean) ----------
def gm_score(y_true, y_pred):
    # labels=[0,1] memastikan matriks konfusi bentuk 2x2 meski ada kelas yg kosong
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # sensitivity
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # specificity
    return float(np.sqrt(tpr * tnr))

# scorer dengan zero_division=0 agar aman saat tidak ada prediksi positif
scoring = {
    "f1":  make_scorer(f1_score, zero_division=0),
    "bas": make_scorer(balanced_accuracy_score),
    "gm":  make_scorer(gm_score),
}

# ---------- CV splitter (group-aware) ----------
def make_group_cv(y, groups, n_splits=5, seed0=RANDOM_STATE, max_tries=50):
    """
    Kembalikan StratifiedGroupKFold yang valid: tiap fold train & valid punya ≥1 positif.
    Jika grup-positif terlalu sedikit, otomatis mengurangi n_splits (minimal 2).
    """
    pos_groups = set(g for g, yy in zip(groups, y) if yy == 1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0 + max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok = True
        for tr, va in cv.split(np.zeros(len(y)), y, groups):
            if (y[tr].sum() == 0) or (y[va].sum() == 0):
                ok = False
                break
        if ok:
            return cv
    # fallback aman
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# ---------- Pipelines TOP-2 ----------
def pipe_rf():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),  # aman untuk sparse
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    ])

def pipe_xgb(scale_pos_weight=1.0):
    return Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')),
    ])

# ---------- Evaluasi: tiap label × 2 model ----------
rows = []
for lab in SELECTED_LABELS:
    y = df_imbalanced[lab].astype(int).values
    cv = make_group_cv(y, GROUPS, n_splits=5, seed0=RANDOM_STATE)
    
    pos_count = np.sum(y == 1)
    neg_count = np.sum(y == 0)
    spw = neg_count / pos_count if pos_count > 0 else 1.0

    models = [
        ("RandomForest", pipe_rf()),
        ("XGBoost",      pipe_xgb(scale_pos_weight=spw)),
    ]

    for name, pipe in models:
        cvres = cross_validate(
            pipe, X, y, groups=GROUPS, cv=cv,
            scoring=scoring, n_jobs=1, return_train_score=False  # n_jobs=1 because RF/XGB use multiple threads
        )
        rows.append({
            "Label": lab,
            "Model": name,
            "F1_mean":  float(np.mean(cvres["test_f1"])),
            "F1_std":   float(np.std(cvres["test_f1"])),
            "BAS_mean": float(np.mean(cvres["test_bas"])),
            "BAS_std":  float(np.std(cvres["test_bas"])),
            "GM_mean":  float(np.mean(cvres["test_gm"])),
            "GM_std":   float(np.std(cvres["test_gm"]))
        })
        print(f"[BASELINE][{lab}][{name}] "
              f"F1={rows[-1]['F1_mean']:.4f}±{rows[-1]['F1_std']:.4f} | "
              f"BAS={rows[-1]['BAS_mean']:.4f}±{rows[-1]['BAS_std']:.4f} | "
              f"GM={rows[-1]['GM_mean']:.4f}±{rows[-1]['GM_std']:.4f}")

res_top2 = pd.DataFrame(rows)

# ---------- Simpan CSV ----------
_ts = datetime.now().strftime("%Y%m%d-%H%M%S")
out_csv = f"baseline_no_resampling_top2_{_ts}.csv"
res_top2.to_csv(out_csv, index=False)
print(f"[STEP7] Hasil baseline (top-2) disimpan: {out_csv}")

# ---------- TAMPILKAN: masing-masing label menampilkan 2 baris (Top-2 models) ----------
def _fmt(ms, ss): return f"{ms:.4f} ± {ss:.4f}"

def show_results_per_label(res_df, label_list):
    for lab in label_list:
        sub = res_df[res_df["Label"] == lab].copy()
        # urutkan dari GM terbaik
        sub = sub.sort_values("GM_mean", ascending=False)
        # format mean±std untuk tampilan
        sub["F1"]  = [_fmt(m, s) for m, s in zip(sub["F1_mean"],  sub["F1_std"])]
        sub["BAS"] = [_fmt(m, s) for m, s in zip(sub["BAS_mean"], sub["BAS_std"])]
        sub["GM"]  = [_fmt(m, s) for m, s in zip(sub["GM_mean"],  sub["GM_std"])]
        view = sub[["Model", "F1", "BAS", "GM"]].reset_index(drop=True)

        print(f"\n=== HASIL PER LABEL: {lab} ===")
        try:
            from IPython.display import display
            display(view)
        except Exception:
            print(view.to_string(index=False))

show_results_per_label(res_top2, SELECTED_LABELS)

# ---------- (Opsional) Ringkasan terbaik per label (berdasarkan GM_mean) ----------
best_by_gm = res_top2.sort_values(["Label","GM_mean"], ascending=[True, False])\
                     .groupby("Label").head(1)[["Label","Model","GM_mean","F1_mean","BAS_mean"]]
best_by_gm = best_by_gm.rename(columns={
    "GM_mean":"GM_best", "F1_mean":"F1_at_bestGM", "BAS_mean":"BAS_at_bestGM"
}).reset_index(drop=True)
print("\n=== RINGKASAN TERBAIK (berdasarkan GM) ===")
try:
    from IPython.display import display
    display(best_by_gm)
except Exception:
    print(best_by_gm.to_string(index=False))

[BASELINE][Product issues][RandomForest] F1=0.0000±0.0000 | BAS=0.5000±0.0000 | GM=0.0000±0.0000
[BASELINE][Product issues][XGBoost] F1=0.0000±0.0000 | BAS=0.4968±0.0017 | GM=0.0000±0.0000
[BASELINE][Skin and subcutaneous tissue disorders][RandomForest] F1=0.9607±0.0081 | BAS=0.5495±0.0223 | GM=0.3160±0.0714
[BASELINE][Skin and subcutaneous tissue disorders][XGBoost] F1=0.9277±0.0065 | BAS=0.6368±0.0330 | GM=0.5731±0.0511
[BASELINE][Nervous system disorders][RandomForest] F1=0.9509±0.0047 | BAS=0.5562±0.0263 | GM=0.3482±0.0858
[BASELINE][Nervous system disorders][XGBoost] F1=0.9196±0.0086 | BAS=0.6493±0.0317 | GM=0.5977±0.0444
[BASELINE][Respiratory, thoracic and mediastinal disorders][RandomForest] F1=0.8277±0.0187 | BAS=0.5451±0.0169 | GM=0.4029±0.0519
[BASELINE][Respiratory, thoracic and mediastinal disorders][XGBoost] F1=0.7755±0.0178 | BAS=0.6040±0.0367 | GM=0.5817±0.0505
[BASELINE][Neoplasms benign, malignant and unspecified (incl cysts and polyps)][RandomForest] F1=0.3917±0.0517

,Model,F1,BAS,GM
0,RandomForest,0.0000 ± 0.0000,0.5000 ± 0.0000,0.0000 ± 0.0000
1,XGBoost,0.0000 ± 0.0000,0.4968 ± 0.0017,0.0000 ± 0.0000



=== HASIL PER LABEL: Skin and subcutaneous tissue disorders ===


,Model,F1,BAS,GM
0,XGBoost,0.9277 ± 0.0065,0.6368 ± 0.0330,0.5731 ± 0.0511
1,RandomForest,0.9607 ± 0.0081,0.5495 ± 0.0223,0.3160 ± 0.0714



=== HASIL PER LABEL: Nervous system disorders ===


...

,Model,F1,BAS,GM
0,RandomForest,0.7217 ± 0.0108,0.6955 ± 0.0135,0.6927 ± 0.0152
1,XGBoost,0.6708 ± 0.0115,0.6607 ± 0.0157,0.6606 ± 0.0157



=== HASIL PER LABEL: Reproductive system and breast disorders ===


,Model,F1,BAS,GM
0,RandomForest,0.6850 ± 0.0087,0.6763 ± 0.0156,0.6750 ± 0.0164
1,XGBoost,0.6732 ± 0.0185,0.6671 ± 0.0234,0.6655 ± 0.0236



=== RINGKASAN TERBAIK (berdasarkan GM) ===


,Label,Model,GM_best,F1_at_bestGM,BAS_at_bestGM
0,Ear and labyrinth disorders,RandomForest,0.627846,0.589390,0.632077
1,Hepatobiliary disorders,RandomForest,0.692675,0.721667,0.695513
2,Immune system disorders,XGBoost,0.578433,0.755508,0.596329
3,"Neoplasms benign, malignant and unspecified (i...",XGBoost,0.594534,0.446486,0.626403
4,Nervous system disorders,XGBoost,0.597723,0.919603,0.649258
5,Product issues,RandomForest,0.000000,0.000000,0.500000
6,Reproductive system and breast disorders,RandomForest,0.674999,0.684989,0.676287
7,"Respiratory, thoracic and mediastinal disorders",XGBoost,0.581700,0.775473,0.603979
8,Skin and subcutaneous tissue disorders,XGBoost,0.573086,0.927693,0.636838


# 9.Try with oversampling the train data

In [ ]:
# -------- Guards --------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df_imbalanced" in globals() and "Groups" in globals(), "df_imbalanced/Groups belum ada (STEP 1 & 3)."
assert "SELECTED_LABELS" in globals(), "SELECTED_LABELS belum ada (STEP 5)."
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 42

# -------- Konfigurasi utama --------
OVER_METHODS = ["SMOTE", "ADASYN" , "RandomOverSampler"]          # metode oversampling yang diuji
MINORITY_MULTIPLIER = 1.30                  # >> 1.0 → minority_after ≈ 1.3 × majority
APPLY_TOMEK = False                         # kalau mau, set True (pembersihan setelah oversampling)
N_SPLITS = 5                                # outer CV splits

# Dua model terbaik untuk evaluasi
def make_model(name):
    if name == "RandomForest":
        # tanpaclass_weight agar efek oversampling terlihat jelas
        return RandomForestClassifier(n_estimators=100, class_weight=None, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "Xgboost":
        return XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    raise ValueError("Unknown model name")

TOP2_MODELS = ["RandomForest", "Xgboost"]

# -------- Metrik --------
def gm_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0.0
    tnr = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return float(np.sqrt(tpr*tnr))

# -------- CV splitter (group-aware, memastikan ada positif) --------
def make_group_cv(y, groups, n_splits=N_SPLITS, seed0=RANDOM_STATE, max_tries=50):
    pos_groups = set(g for g, yy in zip(groups, y) if yy==1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0+max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok = True
        for tr, va in cv.split(np.zeros_like(y), y, groups):
            if (y[tr].sum()==0) or (y[va].sum()==0):
                ok = False; break
        if ok: return cv
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# -------- Oversampling per fold (dinamis, > mayoritas) --------
def oversample_train(X_tr_csr, y_tr, method: str, multiplier: float, use_tomek: bool):
    """
    - Hitung kelas mayoritas/minoritas pada TRAIN fold
    - Standarisasi (fit di TRAIN)
    - Oversample minority hingga target > mayoritas (via dict sampling_strategy)
    - (Opsional) TomekLinks setelah OS
    - Kembalikan: X_os_scaled, y_os, scaler, info_count_before/after
    """
    cnt = Counter(y_tr)
    # kalau 1 kelas: tidak bisa OS algoritmik → skip
    if len(cnt) < 2:
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
        Xs = scaler.fit_transform(Xd)
        return Xs, y_tr, scaler, {"before": cnt, "after": cnt}

    # identifikasi majority/minority
    maj = max(cnt, key=cnt.get)
    minc = min(cnt, key=cnt.get)
    n_maj = cnt[maj]
    n_min = cnt[minc]

    # target minority > majority
    target_min = int(np.ceil(multiplier * n_maj))

    # siapkan data dense & skalakan untuk tetangga
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
    Xs = scaler.fit_transform(Xd)

    # tentukan k tetangga aman (SMOTE/ADASYN butuh >=1)
    k = max(1, min(5, n_min - 1))

    # fallback: kalau n_min < 2, pakai ROS agar tetap bisa “naikkan”
    can_algo = (n_min >= 2)

    try:
        if method.upper() == "SMOTE" and can_algo:
            sampler = SMOTE(sampling_strategy={minc: target_min}, k_neighbors=k, random_state=RANDOM_STATE)
            X_os, y_os = sampler.fit_resample(Xs, y_tr)
        elif method.upper() == "ADASYN" and can_algo:
            sampler = ADASYN(sampling_strategy={minc: target_min}, n_neighbors=k, random_state=RANDOM_STATE)
            X_os, y_os = sampler.fit_resample(Xs, y_tr)
        else:
            ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
            X_os, y_os = ros.fit_resample(Xs, y_tr)
    except ValueError:
        # Fallback if SMOTE/ADASYN fails to generate samples
        ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
        X_os, y_os = ros.fit_resample(Xs, y_tr)

    if use_tomek:
        tl = TomekLinks(n_jobs=-1)
        X_os, y_os = tl.fit_resample(X_os, y_os)

    after_cnt = Counter(y_os)
    info = {"before": cnt, "after": after_cnt, "k_used": k}
    return X_os, y_os, scaler, info

# -------- Evaluasi utama: per label × (SMOTE, ADASYN) × (LinearSVC, Ridge) --------
rows = []
GROUPS = Groups
for lab in SELECTED_LABELS:
    y_all = df_imbalanced[lab].astype(int).values
    cv = make_group_cv(y_all, GROUPS, n_splits=N_SPLITS, seed0=RANDOM_STATE)

    for over in OVER_METHODS:
        for model_name in TOP2_MODELS:
            f1s, bass, gms = [], [], []
            fold_logs = []

            for fold_id, (tr, va) in enumerate(cv.split(np.zeros_like(y_all), y_all, GROUPS), start=1):
                X_tr, X_va = X_INT[tr], X_INT[va]
                y_tr, y_va = y_all[tr], y_all[va]

                # 1) Oversample TRAIN saja (minority > majority)
                X_os, y_os, scaler, info = oversample_train(
                    X_tr, y_tr, method=over, multiplier=MINORITY_MULTIPLIER, use_tomek=APPLY_TOMEK
                )

                # 2) Transform VALID pakai scaler TRAIN
                X_va_scaled = scaler.transform(X_va.toarray() if hasattr(X_va, "toarray") else np.asarray(X_va))

                # 3) Train classifier pada data oversampled (tanpa scaler lagi; sudah scaled)
                clf = make_model(model_name)
                clf.fit(X_os, y_os)

                # 4) Prediksi & metrik
                y_pred = clf.predict(X_va_scaled)
                f1s.append(f1_score(y_va, y_pred, zero_division=0))
                bass.append(balanced_accuracy_score(y_va, y_pred))
                gms.append(gm_score(y_va, y_pred))

                # log per fold (opsional untuk audit)
                fold_logs.append({
                    "Fold": fold_id,
                    "Before_pos": int(info["before"].get(1, 0)),
                    "Before_neg": int(info["before"].get(0, 0)),
                    "After_pos":  int(info["after"].get(1, 0)),
                    "After_neg":  int(info["after"].get(0, 0)),
                    "k_neighbors": info["k_used"],
                })

            # ringkasan per kombinasi
            rows.append({
                "Label": lab,
                "Method_Over": over,
                "Model": model_name,
                "MinorityMultiplier": MINORITY_MULTIPLIER,
                "APPLY_TOMEK": APPLY_TOMEK,
                "F1_mean":  float(np.mean(f1s)),  "F1_std":  float(np.std(f1s)),
                "BAS_mean": float(np.mean(bass)), "BAS_std": float(np.std(bass)),
                "GM_mean":  float(np.mean(gms)),  "GM_std":  float(np.std(gms)),
            })

            # cetak ringkasan + contoh fold 1
            ex = fold_logs[0]
            print(f"[OVER][{lab}][{over}][{model_name}] "
                  f"F1={rows[-1]['F1_mean']:.4f}±{rows[-1]['F1_std']:.4f} | "
                  f"BAS={rows[-1]['BAS_mean']:.4f}±{rows[-1]['BAS_std']:.4f} | "
                  f"GM={rows[-1]['GM_mean']:.4f}±{rows[-1]['GM_std']:.4f} "
                  f"| Fold1 Before(+/−)={ex['Before_pos']}/{ex['Before_neg']} → After(+/−)={ex['After_pos']}/{ex['After_neg']}")

res_over_only = pd.DataFrame(rows)

# -------- Simpan hasil --------
_ts = globals().get("_ts", datetime.now().strftime("%Y%m%d-%H%M%S"))
out_csv = f"oversampling_only_SMOTE_ADASYN_top2_{_ts}.csv"
res_over_only.to_csv(out_csv, index=False)
print(f"[STEP8] Hasil oversampling-only disimpan: {out_csv}")

# -------- Tampilkan ringkasan rapi --------
def fmt(m,s): return f"{m:.4f} ± {s:.4f}"
view = res_over_only.copy()
view["F1"]  = [fmt(m,s) for m,s in zip(view["F1_mean"],  view["F1_std"])]
view["BAS"] = [fmt(m,s) for m,s in zip(view["BAS_mean"], view["BAS_std"])]
view["GM"]  = [fmt(m,s) for m,s in zip(view["GM_mean"],  view["GM_std"])]
view = view[["Label","Method_Over","Model","MinorityMultiplier","APPLY_TOMEK","F1","BAS","GM"]]\
       .sort_values(["Label","Method_Over","GM"], ascending=[True, True, False]).reset_index(drop=True)

try:
    from IPython.display import display
    display(view)
except Exception:
    print(view.to_string(index=False))

[OVER][Product issues][SMOTE][RandomForest] F1=0.0000±0.0000 | BAS=0.4989±0.0014 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Product issues][SMOTE][Xgboost] F1=0.0000±0.0000 | BAS=0.4982±0.0011 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Product issues][ADASYN][RandomForest] F1=0.0000±0.0000 | BAS=0.4986±0.0013 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Product issues][ADASYN][Xgboost] F1=0.0000±0.0000 | BAS=0.4986±0.0013 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Product issues][RandomOverSampler][RandomForest] F1=0.0000±0.0000 | BAS=0.4986±0.0013 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Product issues][RandomOverSampler][Xgboost] F1=0.0000±0.0000 | BAS=0.4972±0.0024 | GM=0.0000±0.0000 | Fold1 Before(+/−)=17/1124 → After(+/−)=1462/1124
[OVER][Skin and subcutaneous tissue disorders][SMOTE][RandomForest] F1=0.9606±0.008

,Label,Method_Over,Model,MinorityMultiplier,APPLY_TOMEK,F1,BAS,GM
0,Ear and labyrinth disorders,ADASYN,RandomForest,1.3,False,0.5815 ± 0.0131,0.6233 ± 0.0213,0.6191 ± 0.0186
1,Ear and labyrinth disorders,ADASYN,Xgboost,1.3,False,0.5586 ± 0.0321,0.6013 ± 0.0224,0.5975 ± 0.0248
2,Ear and labyrinth disorders,RandomOverSampler,RandomForest,1.3,False,0.5992 ± 0.0258,0.6153 ± 0.0204,0.6138 ± 0.0207
3,Ear and labyrinth disorders,RandomOverSampler,Xgboost,1.3,False,0.5718 ± 0.0154,0.5923 ± 0.0209,0.5915 ± 0.0204
4,Ear and labyrinth disorders,SMOTE,RandomForest,1.3,False,0.5780 ± 0.0132,0.6163 ± 0.0145,0.6133 ± 0.0144
5,Ear and labyrinth disorders,SMOTE,Xgboost,1.3,False,0.5619 ± 0.0201,0.6008 ± 0.0166,0.5975 ± 0.0177
6,Hepatobiliary disorders,ADASYN,RandomForest,1.3,False,0.7196 ± 0.0100,0.6934 ± 0.0152,0.6904 ± 0.0171
7,Hepatobiliary disorders,ADASYN,Xgboost,1.3,False,0.6836 ± 0.0097,0.6681 ± 0.0098,0.6675 ± 0.0093
8,Hepatobiliary disorders,RandomOverSampler,RandomForest,1.3,False,0.7037 ± 0.0158,0.6968 ± 0.0211,0.6962 ± 0.0213
9,Hepatobiliary disorders,RandomOverSampler,Xgboost,1.3,False,0.6629 ± 0.0171,0.6634 ± 0.0164,0.6629 ± 0.0162


# 10. Choose the best combination the model through the GM value F1 and BAS tie-Bracker

In [13]:
def load_res_over_only():
    if "res_over_only" in globals() and isinstance(res_over_only, pd.DataFrame):
        return res_over_only.copy()
    candidates = sorted(glob.glob("oversampling_only_SMOTE_ADASYN_RandomOverSampler_*.csv")) + \
                 sorted(glob.glob("/mnt/data/oversampling_only_SMOTE_ADASYN_RandomOverSampler_*.csv"))
    if not candidates:
        raise RuntimeError("Hasil oversampling belum ditemukan. Jalankan STEP 8 dulu.")
    return pd.read_csv(candidates[-1])

res = load_res_over_only()

# Urutkan dgn prioritas: GM_mean ↓, F1_mean ↓, BAS_mean ↓
ranked = res.sort_values(
    ["Label","GM_mean","F1_mean","BAS_mean"],
    ascending=[True, False, False, False]
)

# Ambil 1 terbaik per label
best_per_label = ranked.groupby("Label", as_index=False).head(1).reset_index(drop=True)

# Tabel ringkas untuk dilihat
view = best_per_label[["Label","Method_Over","Model","MinorityMultiplier","APPLY_TOMEK",
                       "GM_mean","F1_mean","BAS_mean"]].copy()
view["GM_mean"]  = view["GM_mean"].round(4)
view["F1_mean"]  = view["F1_mean"].round(4)
view["BAS_mean"] = view["BAS_mean"].round(4)

print("=== Kombinasi Terbaik per Label (berdasarkan GM) ===")
try:
    from IPython.display import display
    display(view)
except Exception:
    print(view.to_string(index=False))

# (Opsional) juga tampilkan TOP-3 per label untuk perbandingan cepat
top3 = ranked.groupby("Label", as_index=False, group_keys=False).head(3)
top3_view = top3[["Label","Method_Over","Model","GM_mean","F1_mean","BAS_mean"]].copy().round(4)
print("\n=== TOP-3 per Label (GM utama) ===")
try:
    from IPython.display import display
    display(top3_view)
except Exception:
    print(top3_view.to_string(index=False))

# (Opsional) kamus pemenang → untuk dipakai di langkah undersampling refinement berikutnya
WINNERS = {
    row["Label"]: {
        "over": row["Method_Over"],
        "model": row["Model"],
        "multiplier": float(row["MinorityMultiplier"]),
        "use_tomek": bool(row["APPLY_TOMEK"])
    }
    for _, row in best_per_label.iterrows()
}
print("\nWINNERS mapping siap dipakai:", WINNERS)

=== Kombinasi Terbaik per Label (berdasarkan GM) ===


,Label,Method_Over,Model,MinorityMultiplier,APPLY_TOMEK,GM_mean,F1_mean,BAS_mean
0,Ear and labyrinth disorders,ADASYN,RandomForest,1.3,False,0.6191,0.5815,0.6233
1,Hepatobiliary disorders,RandomOverSampler,RandomForest,1.3,False,0.6962,0.7037,0.6968
2,Immune system disorders,RandomOverSampler,Xgboost,1.3,False,0.5735,0.7454,0.5893
3,"Neoplasms benign, malignant and unspecified (i...",RandomOverSampler,Xgboost,1.3,False,0.5974,0.4471,0.6265
4,Nervous system disorders,RandomOverSampler,Xgboost,1.3,False,0.5236,0.9303,0.6140
5,Product issues,SMOTE,RandomForest,1.3,False,0.0000,0.0000,0.4989
6,Reproductive system and breast disorders,SMOTE,RandomForest,1.3,False,0.6744,0.6805,0.6757
7,"Respiratory, thoracic and mediastinal disorders",RandomOverSampler,Xgboost,1.3,False,0.5817,0.7688,0.6002
8,Skin and subcutaneous tissue disorders,RandomOverSampler,Xgboost,1.3,False,0.4844,0.9389,0.5956



=== TOP-3 per Label (GM utama) ===


,Label,Method_Over,Model,GM_mean,F1_mean,BAS_mean
38,Ear and labyrinth disorders,ADASYN,RandomForest,0.6191,0.5815,0.6233
40,Ear and labyrinth disorders,RandomOverSampler,RandomForest,0.6138,0.5992,0.6153
36,Ear and labyrinth disorders,SMOTE,RandomForest,0.6133,0.5780,0.6163
46,Hepatobiliary disorders,RandomOverSampler,RandomForest,0.6962,0.7037,0.6968
44,Hepatobiliary disorders,ADASYN,RandomForest,0.6904,0.7196,0.6934
42,Hepatobiliary disorders,SMOTE,RandomForest,0.6867,0.7173,0.6898
35,Immune system disorders,RandomOverSampler,Xgboost,0.5735,0.7454,0.5893
34,Immune system disorders,RandomOverSampler,RandomForest,0.5583,0.7755,0.5909
33,Immune system disorders,ADASYN,Xgboost,0.4776,0.8113,0.5713
29,"Neoplasms benign, malignant and unspecified (i...",RandomOverSampler,Xgboost,0.5974,0.4471,0.6265



WINNERS mapping siap dipakai: {'Ear and labyrinth disorders': {'over': 'ADASYN', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 'Hepatobiliary disorders': {'over': 'RandomOverSampler', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 'Immune system disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 'Neoplasms benign, malignant and unspecified (incl cysts and polyps)': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 'Nervous system disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 'Product issues': {'over': 'SMOTE', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 'Reproductive system and breast disorders': {'over': 'SMOTE', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 'Respiratory, thoracic and mediastinal disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multi

# 11 . HHO apply udersampling the data 


In [ ]:
# =======================================
# STEP 9 — HHO Undersampling Refinement
# (Outer Group-CV; Inner Group-CV utk fitness)
# =======================================
# -------- Guards --------
assert "X_INT" in globals(), "X_INT belum ada (jalankan STEP 4)."
assert "df_imbalanced" in globals() and "Groups" in globals(), "df_imbalanced/Groups belum ada (STEP 1 & 3)."
assert "SELECTED_LABELS" in globals(), "SELECTED_LABELS belum ada (STEP 5)."
try:
    RANDOM_STATE
except NameError:
    RANDOM_STATE = 116

# -------- Pemenang oversampling × model per label (pakai hasilmu) --------
# Kamu bisa ubah per label bila perlu.
WINNERS = globals().get("WINNERS", {
    'Ear and labyrinth disorders': {'over': 'ADASYN', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 
    'Hepatobiliary disorders': {'over': 'RandomOverSampler', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 
    'Immune system disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 
    'Neoplasms benign, malignant and unspecified (incl cysts and polyps)': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 
    'Nervous system disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 
    'Product issues': {'over': 'SMOTE', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 
    'Reproductive system and breast disorders': {'over': 'SMOTE', 'model': 'RandomForest', 'multiplier': 1.3, 'use_tomek': False}, 
    'Respiratory, thoracic and mediastinal disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}, 
    'Skin and subcutaneous tissue disorders': {'over': 'RandomOverSampler', 'model': 'Xgboost', 'multiplier': 1.3, 'use_tomek': False}
})

# -------- Metrik utama --------
def gm_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    tpr = tp/(tp+fn) if (tp+fn)>0 else 0.0
    tnr = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return float(np.sqrt(tpr*tnr))

def score_triple(y_true, y_pred) -> Tuple[float,float,float]:
    return (
        f1_score(y_true, y_pred, zero_division=0),
        balanced_accuracy_score(y_true, y_pred),
        gm_score(y_true, y_pred),
    )

# -------- CV helpers --------
def make_group_cv(y, groups, n_splits=5, seed0=RANDOM_STATE, max_tries=50):
    pos_groups = set(g for g, yy in zip(groups, y) if yy==1)
    if len(pos_groups) < n_splits:
        n_splits = max(2, len(pos_groups))
    for sd in range(seed0, seed0+max_tries):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=sd)
        ok=True
        for tr, va in cv.split(np.zeros_like(y), y, groups):
            if (y[tr].sum()==0) or (y[va].sum()==0):
                ok=False; break
        if ok: return cv
    return StratifiedGroupKFold(n_splits=max(2, min(3, n_splits)), shuffle=True, random_state=seed0)

# -------- Model factory (tanpa class_weight agar efek OS terlihat) --------
def make_model(name):
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=100, class_weight=None, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "XGBoost" or name == "Xgboost":
        return XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    raise ValueError(f"Unknown model name: {name}")

# -------- Oversampling di TRAIN (minority > majority) --------
def oversample_after_subset(X_tr_csr, y_tr, method: str, multiplier: float, use_tomek: bool, k_max=5):
    cnt = Counter(y_tr)
    if len(cnt) < 2:
        # no oversampling possible, just scale
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
        Xs = scaler.fit_transform(Xd)
        return Xs, y_tr, scaler, {"before": cnt, "after": cnt, "k_used": 0}

    maj = max(cnt, key=cnt.get); minc = min(cnt, key=cnt.get)
    n_maj, n_min = cnt[maj], cnt[minc]
    target_min = int(np.ceil(multiplier * n_maj))

    scaler = StandardScaler(with_mean=True, with_std=True)
    Xd = X_tr_csr.toarray() if hasattr(X_tr_csr, "toarray") else np.asarray(X_tr_csr)
    Xs = scaler.fit_transform(Xd)

    k = max(1, min(k_max, n_min - 1))
    can_algo = (n_min >= 2)

    try:
        if method.upper() == "SMOTE" and can_algo:
            sampler = SMOTE(sampling_strategy={minc: target_min}, k_neighbors=k, random_state=RANDOM_STATE)
            X_os, y_os = sampler.fit_resample(Xs, y_tr)
        elif method.upper() == "ADASYN" and can_algo:
            sampler = ADASYN(sampling_strategy={minc: target_min}, n_neighbors=k, random_state=RANDOM_STATE)
            X_os, y_os = sampler.fit_resample(Xs, y_tr)
        else:
            ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
            X_os, y_os = ros.fit_resample(Xs, y_tr)
    except ValueError:
        # Fallback if SMOTE/ADASYN fails to generate samples
        ros = RandomOverSampler(sampling_strategy={minc: target_min}, random_state=RANDOM_STATE)
        X_os, y_os = ros.fit_resample(Xs, y_tr)

    if use_tomek:
        tl = TomekLinks(n_jobs=-1)
        X_os, y_os = tl.fit_resample(X_os, y_os)

    return X_os, y_os, scaler, {"before": cnt, "after": Counter(y_os), "k_used": k}

# -------- Hardness ranking: pilih majority “paling sulit” --------
def hardness_rank_majority(X_csr, y, take_n, model_name="RandomForest"):
    Xd = X_csr.astype(np.float32)
    # Latih classifier balanced utk dapat margin probabilistik
    if model_name.lower() == "xgboost":
        spw = np.sum(y == 0) / np.sum(y == 1) if np.sum(y == 1) > 0 else 1.0
        base = XGBClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
        base.fit(Xd, y)
        scores = base.predict_proba(Xd)[:, 1] - 0.5
    else:
        base = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        base.fit(Xd, y)
        scores = base.predict_proba(Xd)[:, 1] - 0.5
    
    # majority = label 0; paling sulit = margin dekat 0 (|score| kecil) namun di sisi 0
    idx_major = np.where(y == 0)[0]
    margins = np.abs(scores[idx_major])
    # ambil indeks dengan margin terkecil
    order = np.argsort(margins)
    take_n = int(min(len(order), max(1, np.floor(take_n))))
    keep_idx_major = idx_major[order[:take_n]]
    return keep_idx_major

# -------- HHO (sederhana; 1D: keep_ratio) --------
@dataclass
class HHOConfig:
    pop_size: int = 12
    iters: int = 25
    r_min: float = 0.2
    r_max: float = 0.9

def run_hho_optimize_keep_ratio(X_tr, y_tr, groups_tr, over_method, multiplier, model_name,
                                inner_splits=3, cfg=HHOConfig()):
    rng = np.random.RandomState(RANDOM_STATE)
    # inisialisasi populasi
    pop = rng.uniform(cfg.r_min, cfg.r_max, size=cfg.pop_size)
    fitness = np.full(cfg.pop_size, -np.inf, dtype=float)

    def fitness_of_ratio(ratio: float) -> Tuple[float, Dict]:
        # hitung berapa majority yang disimpan
        n_maj = int(np.sum(y_tr == 0))
        keep_n = int(np.ceil(ratio * n_maj))
        # pilih majority paling sulit
        keep_idx_major = hardness_rank_majority(X_tr, y_tr, keep_n, model_name=model_name)
        # bentuk subset train
        keep_mask = np.zeros_like(y_tr, dtype=bool)
        keep_mask[keep_idx_major] = True
        keep_mask |= (y_tr == 1)  # minority semuanya dipakai
        X_sub = X_tr[keep_mask]
        y_sub = y_tr[keep_mask]
        g_sub = groups_tr[keep_mask]

        # inner-CV untuk evaluasi fitness (GM utama; tie F1→BAS)
        cv_inner = make_group_cv(y_sub, g_sub, n_splits=min(inner_splits, 5), seed0=RANDOM_STATE)
        f1s, bass, gms = [], [], []
        for tr, va in cv_inner.split(np.zeros_like(y_sub), y_sub, g_sub):
            X_i_tr, X_i_va = X_sub[tr], X_sub[va]
            y_i_tr, y_i_va = y_sub[tr], y_sub[va]

            # oversample setelah subset (minority > majority)
            X_os, y_os, scaler, _ = oversample_after_subset(
                X_i_tr, y_i_tr, method=over_method, multiplier=multiplier, use_tomek=False
            )
            X_va_scaled = scaler.transform(X_i_va.toarray() if hasattr(X_i_va, "toarray") else np.asarray(X_i_va))

            clf = make_model(model_name)
            clf.fit(X_os, y_os)
            y_hat = clf.predict(X_va_scaled)

            f1, bas, gm = score_triple(y_i_va, y_hat)
            f1s.append(f1); bass.append(bas); gms.append(gm)

        # fitness: GM mean; tie-breakers
        return (float(np.mean(gms)), {
            "gm": float(np.mean(gms)), "f1": float(np.mean(f1s)), "bas": float(np.mean(bass)),
            "keep_ratio": float(ratio), "keep_n": int(keep_n)
        })

    # evaluasi awal
    best_val, best_meta = -np.inf, None
    for i in range(cfg.pop_size):
        val, meta = fitness_of_ratio(pop[i])
        fitness[i] = val
        if val > best_val:
            best_val, best_meta = val, meta

    # iterasi HHO (sederhana 1D)
    for t in range(1, cfg.iters+1):
        E = 2 * (1 - t / cfg.iters)  # energy factor
        for i in range(cfg.pop_size):
            r = pop[i]
            q = rng.rand()
            if abs(E) >= 1:  # exploration
                r_new = best_meta["keep_ratio"] + rng.uniform(-1,1) * abs(best_meta["keep_ratio"] - r)
            else:            # exploitation
                if q >= 0.5:
                    r_new = best_meta["keep_ratio"] - E * abs(best_meta["keep_ratio"] - r)
                else:
                    r_new = best_meta["keep_ratio"] + E * abs(best_meta["keep_ratio"] - r)
            # clamp
            r_new = float(np.clip(r_new, cfg.r_min, cfg.r_max))
            # evaluate
            val, meta = fitness_of_ratio(r_new)
            # greedy accept
            if val > fitness[i]:
                pop[i] = r_new
                fitness[i] = val
                if val > best_val:
                    best_val, best_meta = val, meta

    return best_meta  # {"gm","f1","bas","keep_ratio","keep_n"}

# -------- Execution & Orchestration Functions --------

def evaluate_single_label(lab, X_all, y_all, groups, config, hho_cfg, n_splits=5):
    """Proses evaluasi outer CV untuk satu dataset/label."""
    cv_outer = make_group_cv(y_all, groups, n_splits=n_splits, seed0=RANDOM_STATE)
    
    over_m, model_m = config["over"], config["model"]
    mult_m, use_tomek = float(config["multiplier"]), bool(config.get("use_tomek", False))
    
    f1s, bass, gms = [], [], []
    folds_meta = []
    
    for fold_id, (tr, va) in enumerate(cv_outer.split(np.zeros_like(y_all), y_all, groups), start=1):
        X_tr, X_va = X_all[tr], X_all[va]
        y_tr, y_va = y_all[tr], y_all[va]
        g_tr = groups[tr]

        # 1) HHO: cari keep_ratio terbaik di TRAIN (inner-CV)
        best_meta = run_hho_optimize_keep_ratio(
            X_tr, y_tr, g_tr, over_method=over_m, multiplier=mult_m, model_name=model_m,
            inner_splits=3, cfg=hho_cfg
        )

        # 2) Bangun TRAIN final dengan keep_ratio terbaik
        keep_n = best_meta["keep_n"]
        keep_idx_major = hardness_rank_majority(X_tr, y_tr, keep_n, model_name=model_m)
        keep_mask = np.zeros_like(y_tr, dtype=bool)
        keep_mask[keep_idx_major] = True
        keep_mask |= (y_tr == 1)
        X_sub, y_sub = X_tr[keep_mask], y_tr[keep_mask]

        # 3) Oversampling (minority > majority), optional Tomek
        X_os, y_os, scaler, info = oversample_after_subset(
            X_sub, y_sub, method=over_m, multiplier=mult_m, use_tomek=use_tomek
        )
        X_va_scaled = scaler.transform(X_va.toarray() if hasattr(X_va, "toarray") else np.asarray(X_va))

        # 4) Train final & eval di VALID outer
        clf = make_model(model_m)
        clf.fit(X_os, y_os)
        y_hat = clf.predict(X_va_scaled)

        f1, bas, gm = score_triple(y_va, y_hat)
        f1s.append(f1); bass.append(bas); gms.append(gm)

        folds_meta.append({
            "Fold": fold_id,
            "Best_keep_ratio": best_meta["keep_ratio"],
            "Best_inner_GM": best_meta["gm"],
            "Train_before_pos": int(np.sum(y_tr==1)),
            "Train_before_neg": int(np.sum(y_tr==0)),
            "Train_after_keep_neg": int(keep_n),
            "OS_after_pos": int(info["after"].get(1,0)),
            "OS_after_neg": int(info["after"].get(0,0)),
        })

        print(f"[HHO][{lab}] Fold{fold_id}: keep_ratio={best_meta['keep_ratio']:.3f} | "
              f"innerGM={best_meta['gm']:.4f} | "
              f"F1={f1:.4f} BAS={bas:.4f} GM={gm:.4f} | "
              f"before(+/−)={np.sum(y_tr==1)}/{np.sum(y_tr==0)} → keep_neg={keep_n} → "
              f"OS(+/−)={info['after'].get(1,0)}/{info['after'].get(0,0)}")

    return {
        "Method_Over": over_m,
        "Label": lab,
        "Under": "HHO",
        "Valid_GM": float(np.mean(gms)),
        "Valid_BAS": float(np.mean(bass)),
        "Valid_F1": float(np.mean(f1s)),
        "Groups": int(len(np.unique(groups))),
        "BestFitness(innerGM)_mean": float(np.mean([m["Best_inner_GM"] for m in folds_meta])),
        "Best_keep_ratio_mean": float(np.mean([m["Best_keep_ratio"] for m in folds_meta])),
        "MinorityMultiplier": mult_m,
        "APPLY_TOMEK": use_tomek,
        "Model": model_m
    }

def run_all_labels_hho(labels, df_data, X_data, groups, winners_dict, n_splits=5):
    """Menjalankan evaluasi HHO pada semua label terpilih secara berurutan."""
    rows = []
    # Anda juga bisa mengekspor konfigurasi HHO ini menjadi parameter masukan jika diinginkan
    hho_cfg = HHOConfig(pop_size=12, iters=25, r_min=0.2, r_max=0.9)
    
    for lab in labels:
        y_all = df_data[lab].astype(int).values
        # fallback config per label jika tidak ada dalam pemenang
        win_config = winners_dict.get(lab, {"over": "ADASYN", "model": "RandomForest", "multiplier": 1.30, "use_tomek": False})
        
        label_res = evaluate_single_label(
            lab=lab, X_all=X_data, y_all=y_all, groups=groups, 
            config=win_config, hho_cfg=hho_cfg, n_splits=n_splits
        )
        rows.append(label_res)
        
    return pd.DataFrame(rows)

def save_and_display_hho_results(res_df):
    """Menyimpan hasil eksekusi dataframe ke CSV dan merendernya dalam bentuk tabel."""
    _ts = globals().get("_ts", datetime.now().strftime("%Y%m%d-%H%M%S"))
    out_csv = f"hho_refinement_results_{_ts}.csv"
    res_df.to_csv(out_csv, index=False)
    print(f"[STEP9] Hasil HHO refinement disimpan: {out_csv}\n")

    def fmt(x): return f"{x:.4f}"
    view = res_df.copy()
    for col in ["Valid_GM", "Valid_BAS", "Valid_F1", "BestFitness(innerGM)_mean"]:
        if col in view.columns:
            view[col] = view[col].map(fmt)
            
    view = view[["Method_Over","Label","Under","Model","MinorityMultiplier",
                 "Best_keep_ratio_mean","BestFitness(innerGM)_mean",
                 "Valid_GM","Valid_BAS","Valid_F1","APPLY_TOMEK"]]
    try:
        from IPython.display import display
        display(view)
    except Exception:
        print(view.to_string(index=False))

# -------- Jalankan Pipeline --------
res_hho = run_all_labels_hho(
    labels=SELECTED_LABELS,
    df_data=df_imbalanced,
    X_data=X_INT,
    groups=Groups,
    winners_dict=WINNERS,
    n_splits=5
)
save_and_display_hho_results(res_hho)

[HHO][Product issues] Fold1: keep_ratio=0.463 | innerGM=0.0000 | F1=0.0000 BAS=0.4982 GM=0.0000 | before(+/−)=17/1124 → keep_neg=520 → OS(+/−)=676/520
[HHO][Product issues] Fold2: keep_ratio=0.463 | innerGM=0.0000 | F1=0.0000 BAS=0.5000 GM=0.0000 | before(+/−)=20/1121 → keep_neg=519 → OS(+/−)=675/519
[HHO][Product issues] Fold3: keep_ratio=0.463 | innerGM=0.0000 | F1=0.0000 BAS=0.5000 GM=0.0000 | before(+/−)=13/1129 → keep_neg=523 → OS(+/−)=680/523
[HHO][Product issues] Fold4: keep_ratio=0.463 | innerGM=0.0000 | F1=0.0000 BAS=0.4964 GM=0.0000 | before(+/−)=18/1124 → keep_neg=520 → OS(+/−)=676/520
[HHO][Product issues] Fold5: keep_ratio=0.463 | innerGM=0.0000 | F1=0.0000 BAS=0.5000 GM=0.0000 | before(+/−)=20/1122 → keep_neg=520 → OS(+/−)=676/520
[HHO][Skin and subcutaneous tissue disorders] Fold1: keep_ratio=0.850 | innerGM=0.4694 | F1=0.9480 BAS=0.5660 GM=0.4208 | before(+/−)=1048/93 → keep_neg=80 → OS(+/−)=1048/1363
[HHO][Skin and subcutaneous tissue disorders] Fold2: keep_ratio=0.834

ValueError: No samples will be generated with the provided ratio settings.